# CajaClara — Capa de recomendación (motor de reglas)
**Hackathon BBVA 2026 — Equipo 7**
**Scope: IA / ML / analítica (25%)**

Los otros 2 notebooks predicen (`modelo_ingreso_egreso_lightgbm.ipynb`) y clasifican riesgo
(`credit_scoring_bodegas_logreg.ipynb`), pero **ninguno recomienda una acción**. Este notebook
cierra ese hueco: toma los outputs de ambos modelos + las features ya calculadas, y genera
**recomendaciones accionables y explicables** — no solo un número.

## Por qué reglas y no otro modelo de ML

- Encaja directo con el esquema ya definido por el equipo: la tabla `alertas` (modelo_datos.dbml)
  ya tiene `tipo_alerta`, `prioridad`, `accion_sugerida` y `causas` — este motor llena exactamente
  esos campos.
- Es **determinístico y explicable**: cada recomendación cita la(s) variable(s) que la dispararon
  (`causas`), igual que el baseline de `riesgo.py` del backend (ellos también prefieren reglas
  trazables sobre cajas negras cuando no hay ML integrado).
- No requiere cargar ningún `.joblib` adicional: solo consume los outputs que los otros 2
  notebooks ya producen (`ingreso_pred_*`, `egreso_pred_*`, `score`), así que es liviano y fácil
  de integrar sin agregar dependencias nuevas al backend.

## Qué resuelve cada regla

| `tipo_alerta` (ya existe en el enum) | Prioridad | Dispara cuando... | `accion_sugerida` |
|---|---|---|---|
| `faltante_caja` | 1 | El saldo proyectado (P10) cae bajo el mínimo operativo en los próximos 14 días | `ajustar_pedido` |
| `gasto_inusual` | 5 | El egreso proyectado (P50) supera significativamente su propio promedio histórico | `ver_por_que` |
| `caida_ventas` | 6 | Regularidad de ventas baja (`pct_dias_con_venta` bajo o `cv_ventas` alto) | `ver_por_que` |
| `fiado_vencido` | 7 | Fía mucho (`pct_fiado_sobre_ventas` alto) y cobra poco a tiempo | `recordar_cliente` |
| `meta_lograda` | 8 | Score de salud financiera en zona verde (>=70) | — (refuerzo positivo) |

No se inventan tipos nuevos: los 5 ya existen en `Enum tipo_alerta` del modelo de datos.


In [1]:
import json
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)


## 1. Umbrales de las reglas

Documentados aquí explícitamente (trazabilidad): cada umbral es un supuesto razonable para el
MVP, ajustable con data real sin tocar la lógica del motor.


In [2]:
UMBRALES = {
    "dias_proyeccion_faltante_caja": 14,
    "cv_ventas_alto": 0.5,            # por encima de esto, ventas muy irregulares
    "pct_dias_con_venta_bajo": 0.6,   # por debajo de esto, caida de ventas
    "pct_fiado_sobre_ventas_alto": 0.35,
    "pct_fiado_cobrado_a_tiempo_bajo": 0.5,
    "gasto_inusual_factor": 1.5,      # egreso_p50 > factor * egreso_media30 historico
    "score_meta_lograda": 70,
}

PRIORIDAD_TIPO_ALERTA = {  # igual orden que Enum tipo_alerta del modelo de datos
    "faltante_caja": 1, "pedido_excesivo": 2, "retiro_excesivo": 3, "obligacion_proxima": 4,
    "gasto_inusual": 5, "caida_ventas": 6, "fiado_vencido": 7, "meta_lograda": 8, "resumen_semanal": 9,
}


## 2. Motor de reglas

Cada regla es una función pura: recibe un `contexto` (dict con los outputs de forecast + credit
scoring + features) y devuelve `None` si no aplica, o un dict listo para insertar en `alertas`
si aplica. `causas` sigue el formato ya definido en el modelo de datos:
`[{"causa": "...", "impacto": valor}]`.


In [3]:
def regla_faltante_caja(ctx):
    saldo_p10 = ctx.get("saldo_proyectado_p10")
    minimo_operativo = ctx.get("minimo_operativo")
    if saldo_p10 is None or minimo_operativo is None:
        return None
    if saldo_p10 >= minimo_operativo:
        return None
    return {
        "tipo_alerta": "faltante_caja",
        "prioridad": PRIORIDAD_TIPO_ALERTA["faltante_caja"],
        "accion_sugerida": "ajustar_pedido",
        "mensaje": (f"Tu caja podría caer a S/.{saldo_p10:,.0f} en los próximos "
                    f"{UMBRALES['dias_proyeccion_faltante_caja']} días, por debajo de tu mínimo "
                    f"operativo (S/.{minimo_operativo:,.0f}). Considera reducir compras o cobrar fiados."),
        "causas": [{"causa": "saldo_proyectado_bajo_minimo", "impacto": round(minimo_operativo - saldo_p10, 2)}],
    }


def regla_gasto_inusual(ctx):
    egreso_p50 = ctx.get("egreso_pred_p50")
    egreso_media30 = ctx.get("egreso_media30")
    if egreso_p50 is None or not egreso_media30:
        return None
    factor = egreso_p50 / egreso_media30
    if factor < UMBRALES["gasto_inusual_factor"]:
        return None
    return {
        "tipo_alerta": "gasto_inusual",
        "prioridad": PRIORIDAD_TIPO_ALERTA["gasto_inusual"],
        "accion_sugerida": "ver_por_que",
        "mensaje": (f"Tu egreso esperado (S/.{egreso_p50:,.0f}) es {factor:.1f}x tu promedio de los "
                    f"últimos 30 días (S/.{egreso_media30:,.0f}). Revisa si hubo una compra o pago extra."),
        "causas": [{"causa": "egreso_sobre_promedio_30d", "impacto": round(factor, 2)}],
    }


def regla_caida_ventas(ctx):
    pct_dias = ctx.get("pct_dias_con_venta")
    cv = ctx.get("cv_ventas")
    if pct_dias is None or cv is None:
        return None
    if pct_dias >= UMBRALES["pct_dias_con_venta_bajo"] and cv <= UMBRALES["cv_ventas_alto"]:
        return None
    return {
        "tipo_alerta": "caida_ventas",
        "prioridad": PRIORIDAD_TIPO_ALERTA["caida_ventas"],
        "accion_sugerida": "ver_por_que",
        "mensaje": "Tus ventas están siendo irregulares o menos frecuentes de lo usual. Revisa horarios de atención y stock disponible.",
        "causas": [
            {"causa": "pct_dias_con_venta_bajo", "impacto": round(pct_dias, 2)},
            {"causa": "cv_ventas_alto", "impacto": round(cv, 2)},
        ],
    }


def regla_fiado_vencido(ctx):
    pct_fiado = ctx.get("pct_fiado_sobre_ventas")
    pct_cobrado = ctx.get("pct_fiado_cobrado_a_tiempo")
    if pct_fiado is None or pct_cobrado is None:
        return None
    if pct_fiado < UMBRALES["pct_fiado_sobre_ventas_alto"] or pct_cobrado >= UMBRALES["pct_fiado_cobrado_a_tiempo_bajo"]:
        return None
    return {
        "tipo_alerta": "fiado_vencido",
        "prioridad": PRIORIDAD_TIPO_ALERTA["fiado_vencido"],
        "accion_sugerida": "recordar_cliente",
        "mensaje": (f"{pct_fiado*100:.0f}% de tus ventas son fiadas y solo {pct_cobrado*100:.0f}% se cobra "
                    "a tiempo. Considera recordar a tus clientes o reducir el fiado."),
        "causas": [
            {"causa": "pct_fiado_sobre_ventas_alto", "impacto": round(pct_fiado, 2)},
            {"causa": "pct_fiado_cobrado_a_tiempo_bajo", "impacto": round(pct_cobrado, 2)},
        ],
    }


def regla_meta_lograda(ctx):
    score = ctx.get("score")
    if score is None or score < UMBRALES["score_meta_lograda"]:
        return None
    return {
        "tipo_alerta": "meta_lograda",
        "prioridad": PRIORIDAD_TIPO_ALERTA["meta_lograda"],
        "accion_sugerida": None,
        "mensaje": f"¡Tu score de salud financiera es {score}! Vas por buen camino, sigue registrando tus movimientos.",
        "causas": [{"causa": "score_en_zona_verde", "impacto": score}],
    }


REGLAS = [
    regla_faltante_caja,
    regla_gasto_inusual,
    regla_caida_ventas,
    regla_fiado_vencido,
    regla_meta_lograda,
]


def generar_recomendaciones(ctx, max_recomendaciones=3):
    """Evalua todas las reglas y devuelve las que aplican, ordenadas por prioridad (1 = mas urgente)."""
    resultados = [r(ctx) for r in REGLAS]
    resultados = [r for r in resultados if r is not None]
    resultados.sort(key=lambda r: r["prioridad"])
    return resultados[:max_recomendaciones]


## 3. Demo sobre bodegas sintéticas

Se simula el `contexto` combinando: features del notebook de credit scoring (regularidad,
fiado), un egreso proyectado (simulando el output del notebook de forecast), el score (output
del notebook de credit scoring), y un saldo/mínimo operativo de ejemplo (vendrían de `empresas`).


In [4]:
N_DEMO = 8

contextos = []
for i in range(N_DEMO):
    pct_dias = float(np.clip(rng.normal(0.8, 0.2), 0.2, 1.0))
    cv_ventas = float(np.clip(rng.normal(0.35, 0.25), 0.05, 1.3))
    pct_fiado = float(np.clip(rng.normal(0.25, 0.2), 0, 0.7))
    pct_cobrado = float(np.clip(rng.normal(0.65, 0.25), 0, 1))
    score = int(np.clip(rng.normal(55, 25), 5, 99))
    egreso_media30 = float(rng.lognormal(5.0, 0.4))
    egreso_p50 = egreso_media30 * float(np.clip(rng.normal(1.1, 0.5), 0.5, 3.0))
    minimo_operativo = float(rng.lognormal(6.0, 0.3))
    saldo_p10 = minimo_operativo * float(np.clip(rng.normal(1.1, 0.6), 0.2, 2.5))

    contextos.append({
        "bodega_id": f"BOD-{i:05d}",
        "pct_dias_con_venta": pct_dias,
        "cv_ventas": cv_ventas,
        "pct_fiado_sobre_ventas": pct_fiado,
        "pct_fiado_cobrado_a_tiempo": pct_cobrado,
        "score": score,
        "egreso_media30": egreso_media30,
        "egreso_pred_p50": egreso_p50,
        "minimo_operativo": minimo_operativo,
        "saldo_proyectado_p10": saldo_p10,
    })

filas = []
for ctx in contextos:
    recos = generar_recomendaciones(ctx)
    for r in recos:
        filas.append({"bodega_id": ctx["bodega_id"], **r})

recomendaciones_demo = pd.DataFrame(filas)
recomendaciones_demo


,bodega_id,tipo_alerta,prioridad,accion_sugerida,mensaje,causas
0,BOD-00001,faltante_caja,1,ajustar_pedido,Tu caja podría caer a S/.236 en los próximos 1...,"[{'causa': 'saldo_proyectado_bajo_minimo', 'im..."
1,BOD-00001,caida_ventas,6,ver_por_que,Tus ventas están siendo irregulares o menos fr...,"[{'causa': 'pct_dias_con_venta_bajo', 'impacto..."
2,BOD-00001,meta_lograda,8,NaN,¡Tu score de salud financiera es 83! Vas por b...,"[{'causa': 'score_en_zona_verde', 'impacto': 83}]"
3,BOD-00002,meta_lograda,8,NaN,¡Tu score de salud financiera es 85! Vas por b...,"[{'causa': 'score_en_zona_verde', 'impacto': 85}]"
4,BOD-00004,meta_lograda,8,NaN,¡Tu score de salud financiera es 73! Vas por b...,"[{'causa': 'score_en_zona_verde', 'impacto': 73}]"
5,BOD-00005,faltante_caja,1,ajustar_pedido,Tu caja podría caer a S/.237 en los próximos 1...,"[{'causa': 'saldo_proyectado_bajo_minimo', 'im..."
6,BOD-00005,caida_ventas,6,ver_por_que,Tus ventas están siendo irregulares o menos fr...,"[{'causa': 'pct_dias_con_venta_bajo', 'impacto..."
7,BOD-00007,faltante_caja,1,ajustar_pedido,Tu caja podría caer a S/.116 en los próximos 1...,"[{'causa': 'saldo_proyectado_bajo_minimo', 'im..."
8,BOD-00007,caida_ventas,6,ver_por_que,Tus ventas están siendo irregulares o menos fr...,"[{'causa': 'pct_dias_con_venta_bajo', 'impacto..."


## 4. Exportación (reglas versionadas)

A diferencia de los otros 2 notebooks, aquí no hay pesos de ML que exportar — se exporta el
**config de umbrales y prioridades** como JSON versionado, para que el backend pueda aplicar
exactamente las mismas reglas sin reimplementar la lógica (y para poder ajustar umbrales sin
tocar código, solo el JSON).


In [5]:
import os

MOTOR_VERSION = "reglas_recomendacion-1.0"

ARTIFACTS_DIR = os.path.join("artifacts", "motor_recomendaciones")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

config = {
    "motor_version": MOTOR_VERSION,
    "umbrales": UMBRALES,
    "prioridad_tipo_alerta": PRIORIDAD_TIPO_ALERTA,
    "reglas_implementadas": [
        {"tipo_alerta": "faltante_caja", "requiere": ["saldo_proyectado_p10 (de predicciones_saldo)", "empresas.minimo_operativo"]},
        {"tipo_alerta": "gasto_inusual", "requiere": ["egreso_pred_p50 (forecast)", "egreso_media30 (feature del forecast)"]},
        {"tipo_alerta": "caida_ventas", "requiere": ["pct_dias_con_venta, cv_ventas (features credit scoring)"]},
        {"tipo_alerta": "fiado_vencido", "requiere": ["pct_fiado_sobre_ventas, pct_fiado_cobrado_a_tiempo (features credit scoring)"]},
        {"tipo_alerta": "meta_lograda", "requiere": ["score (output credit scoring)"]},
    ],
    "notas": "Motor 100% basado en reglas deterministicas (no ML): consume los outputs de "
             "modelo_ingreso_egreso_lightgbm.ipynb y credit_scoring_bodegas_logreg.ipynb, no requiere "
             "cargar ningun .joblib adicional. Llena directo los campos tipo_alerta, prioridad, "
             "accion_sugerida y causas de la tabla alertas (modelo_datos.dbml).",
}
with open(os.path.join(ARTIFACTS_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"Config del motor exportado a {ARTIFACTS_DIR}/")
print(os.listdir(ARTIFACTS_DIR))


Config del motor exportado a artifacts\motor_recomendaciones/
['config.json']


## 5. Resumen ejecutivo

- **Qué resuelve**: convierte las predicciones de los otros 2 notebooks en **acciones concretas**
  (`accion_sugerida`) con su **justificación trazable** (`causas`), cerrando el gap de "solo
  predicción, no recomendación" señalado en la evaluación del equipo.
- **Por qué reglas y no otro modelo**: determinismo, explicabilidad, cero dependencias nuevas, y
  encaje directo con la tabla `alertas` ya diseñada por el equipo.
- **Integración**: el backend llama `generar_recomendaciones(contexto)` con los campos que ya
  tiene disponibles de los otros 2 modelos (o del baseline, si el ML no está activo todavía) y
  escribe el resultado directo en `alertas`.
- **Extensible**: agregar una regla nueva es agregar una función a `REGLAS` — no rompe las
  existentes ni requiere reentrenar nada.
